In [ ]:
# Packages and data

library(tidyverse)
library(scales)
library(systemfonts)
library(ragg)

dc_births <- read_csv("data/processed/dc-births-1995-2025.csv")

# Helper to save plots as high-res PNGs with golden-ratio aspect ratio
rlsave <- function(...) {
  ggsave(
    device = ragg::agg_png(),
    width = 1000,
    height = 1000 / ((1 + sqrt(5)) / 2),
    scale = 2,
    units = "px",
    dpi = "retina",
    ...
  )
}

# Custom ggplot theme built on theme_minimal
theme_rory <- theme_minimal(
  base_size = 12,
  base_family = "Lato-Regular",
  header_family = "PlayfairDisplayRoman-SemiBold",
  paper = "#fbf5f5",
  ink = "#070a0c",
  accent = "#b46466"
) +
  theme(
    plot.title.position = "plot",
    title = element_text(size = 14),
    axis.title = element_text(size = 12),
    axis.text = element_text(size = 10)
  )

set_theme(theme_rory)


In [ ]:
# Compute y-axis limits: pad by 1/3 of the data range above and below,
# then round to nearest thousand for clean axis breaks
ymax <- max(dc_births$births)
ymin <- min(dc_births$births)
span <- ymax - ymin

yupper <- round(ymax + (1 / 3) * span, -3)
ylower <- round(ymin - (1 / 3) * span, -3)

births_plot <- dc_births |>
  ggplot(aes(x = as.integer(year_code), y = births, group = 1)) +
  geom_smooth(
    method = "loess",
    color = "#339589",
    fill = "#b8b5a8",
    lineend = "round"
  ) +
  geom_line(
    linewidth = 1,
    color = "#335195",
    lineend = "round",
    linejoin = "round"
  ) +
  labs(
    title = "Births in the District of Columbia (1995-2025)",
    x = "Year",
    y = "Births"
  ) +
  scale_x_continuous(
    breaks = seq.int(1995, 2025, 5)
  ) +
  scale_y_continuous(
    breaks = seq.int(ylower, yupper, 1000),
    limits = c(ylower, yupper),
    expand = expansion(c(0, 0)),
    labels = label_number(scale_cut = cut_short_scale())
  ) +
  theme_sub_panel(
    grid.major.x = element_blank(),
    grid.minor = element_blank()
  )

rlsave(
  filename = "outputs/dc-births-95-25.png",
  plot = births_plot
)

births_plot


# Forecasting DC Births 2025–2029

Fit ARIMA and ETS models on the 1995–2024 final birth counts and forecast 5 years ahead. We compare both models and plot the ensemble mean alongside 80 % and 95 % prediction intervals.

In [ ]:
# Packages
library(tidyverse)
library(fable)
library(tsibble)
library(fabletools)
library(scales)
library(ragg)

dc_births <- read_csv("data/processed/dc-births-1995-2025.csv")

# Use only final (non-provisional) data for model training
births_train <- dc_births |>
  filter(status == "final") |>
  select(year = year_code, births) |>
  as_tsibble(index = year)

births_train

In [ ]:
# Fit ARIMA and ETS models, then form a combination model
fit <- births_train |>
  model(
    arima = ARIMA(births),
    ets   = ETS(births)
  )

# Report model selections
report(fit |> select(arima))
report(fit |> select(ets))

In [ ]:
# Forecast 5 years: 2025–2029
fc <- fit |> forecast(h = 5)

# Extract tidy forecast table with 80% and 95% intervals
fc_tbl <- fc |>
  hilo(level = c(80, 95)) |>
  unpack_hilo(c(`80%`, `95%`)) |>
  as_tibble() |>
  rename(
    lower80 = `80%_lower`,
    upper80 = `80%_upper`,
    lower95 = `95%_lower`,
    upper95 = `95%_upper`,
    forecast_mean = .mean
  )

fc_tbl

In [ ]:
# Historical data for the plot backdrop
historical <- dc_births |>
  filter(status == "final") |>
  select(year = year_code, births)

# Colour palette aligned with the existing notebook style
arima_col <- "#b46466"  # accent red
ets_col   <- "#339589"  # teal

# Compute shared y-axis limits
ymax <- max(historical$births)
ymin <- min(historical$births)
span <- ymax - ymin
yupper <- round(ymax + (1 / 3) * span, -3)
ylower <- round(ymin - (1 / 3) * span, -3)

fc_plot <- ggplot() +
  # 95% prediction bands
  geom_ribbon(
    data = fc_tbl |> filter(.model == "arima"),
    aes(x = year, ymin = lower95, ymax = upper95),
    fill = arima_col, alpha = 0.15
  ) +
  geom_ribbon(
    data = fc_tbl |> filter(.model == "ets"),
    aes(x = year, ymin = lower95, ymax = upper95),
    fill = ets_col, alpha = 0.15
  ) +
  # 80% prediction bands
  geom_ribbon(
    data = fc_tbl |> filter(.model == "arima"),
    aes(x = year, ymin = lower80, ymax = upper80),
    fill = arima_col, alpha = 0.25
  ) +
  geom_ribbon(
    data = fc_tbl |> filter(.model == "ets"),
    aes(x = year, ymin = lower80, ymax = upper80),
    fill = ets_col, alpha = 0.25
  ) +
  # Historical line
  geom_line(
    data = historical,
    aes(x = year, y = births),
    linewidth = 1, color = "#335195",
    lineend = "round", linejoin = "round"
  ) +
  # Forecast mean lines
  geom_line(
    data = fc_tbl |> filter(.model == "arima"),
    aes(x = year, y = forecast_mean, color = "ARIMA"),
    linewidth = 1, linetype = "dashed",
    lineend = "round"
  ) +
  geom_line(
    data = fc_tbl |> filter(.model == "ets"),
    aes(x = year, y = forecast_mean, color = "ETS"),
    linewidth = 1, linetype = "dashed",
    lineend = "round"
  ) +
  # Vertical line at forecast origin
  geom_vline(xintercept = 2024.5, color = "#070a0c", linewidth = 0.4,
             linetype = "dotted") +
  scale_color_manual(
    name = "Model",
    values = c(ARIMA = arima_col, ETS = ets_col)
  ) +
  scale_x_continuous(breaks = seq(1995, 2029, 5)) +
  scale_y_continuous(
    breaks = seq(ylower, yupper, 1000),
    limits = c(ylower, yupper),
    expand = expansion(c(0, 0)),
    labels = label_number(scale_cut = cut_short_scale())
  ) +
  labs(
    title = "DC Births Forecast 2025–2029",
    subtitle = "ARIMA and ETS models fit on 1995–2024 final data; shading shows 80% and 95% prediction intervals",
    x = "Year",
    y = "Births"
  ) +
  theme(legend.position = "bottom")

rlsave(
  filename = "outputs/dc-births-forecast-2025-2029.png",
  plot = fc_plot
)

fc_plot

In [ ]:
# Summary table: point forecasts and 95% intervals by model
fc_tbl |>
  select(.model, year, forecast_mean, lower95, upper95) |>
  mutate(across(where(is.numeric), round)) |>
  arrange(.model, year) |>
  print(n = Inf)